## SCENARIO 1: Retail Product Catalog Updates
A retail company receives daily updates for its product catalog, including new products, price changes, and discontinued items. Instead of overwriting the entire catalog or simply appending new records, they need to upsert the incoming data—updating existing products with the latest information and inserting new products—ensuring the catalog remains accurate and up-to-date in real-time.

## Querying Source

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *
from pyspark.sql.types import *

# catalog -> schema -> table
df = spark.sql("SELECT * FROM pyspark_catalog.source.products")

id,name,price,category,updatedDate
1,iPhone,1000,phone,2026-08-20T16:33:38.615Z
2,Macbook,2000,laptop,2026-08-20T16:33:38.615Z
3,Macbook Pro,3000,laptop,2026-08-20T16:33:38.615Z
4,Macbook Air,4000,luxury,2026-08-20T16:33:38.615Z
5,Macbook Pro 16,5000,pristine,2026-08-20T16:33:38.615Z
5,Macbook Laptop Pop,5000,pristine,2026-08-21T03:27:50.518Z


## Deduplicate by ID (Keep latest updatedDate)

In [0]:
df = df.withColumn('dedup', row_number().over(Window
                                              .partitionBy('id')
                                              .orderBy(col('updatedDate').desc())))  

df = df.filter(col('dedup') == 1).drop('dedup')         # Filter then drop 

display(df)  

# Deduplicate the source by id using row_number() with partitionBy("id") (partition creates a separate window for each id)
# orderBy(col('updatedDate').desc()) (puts the newest record first within each ID.)

id,name,price,category,updatedDate
1,iPhone,1000,phone,2026-08-20T16:33:38.615Z
2,Macbook,2000,laptop,2026-08-20T16:33:38.615Z
3,Macbook Pro,3000,laptop,2026-08-20T16:33:38.615Z
4,Macbook Air,4000,luxury,2026-08-20T16:33:38.615Z
5,Macbook Laptop Pop,5000,pristine,2026-08-21T03:27:50.518Z


## UPSERT

In [0]:
from delta.tables import DeltaTable

target_path = '/Volumes/pyspark_catalog/source/db_volume/products_sink/'

# Incremental Load
if DeltaTable.isDeltaTable(spark, target_path):

    target = DeltaTable.forPath(spark, target_path)    
    
    # TARGET.merge(SOURCE, condition)
    target.alias('target').merge(
        df.alias('source'),
        "source.id = target.id"
    ).whenMatchedUpdateAll(
        condition="source.updatedDate >= target.updatedDate"
    ).whenNotMatchedInsertAll() \
    .execute()  
    
# Initial Load
else:      
    df.write.format('delta')\
        .mode('overwrite')\
        .save(target_path)

In [0]:
%sql
SELECT * FROM delta.`/Volumes/pyspark_catalog/source/db_volume/products_sink/`

id,name,price,category,updatedDate
1,iPhone,1000,phone,2026-08-20T16:33:38.615Z
2,Macbook,2000,laptop,2026-08-20T16:33:38.615Z
3,Macbook Pro,3000,laptop,2026-08-20T16:33:38.615Z
4,Macbook Air,4000,luxury,2026-08-20T16:33:38.615Z
5,Macbook Laptop Pop,5000,pristine,2026-08-21T03:27:50.518Z
